In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalogo", "ventas")
dbutils.widgets.text("esquema_source", "bronze")
dbutils.widgets.text("esquema_sink", "silver")
dbutils.widgets.text("storageName", "adlsproyecto2026prod")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")
storageName = dbutils.widgets.get("storageName")

In [0]:
# Lectura de tablas Bronze
df_sales = spark.table(f"{catalogo}.{esquema_source}.sales")
df_stores = spark.table(f"{catalogo}.{esquema_source}.stores")


In [0]:
df_transformed = df_sales.alias("s").join(
    df_stores.alias("st"),
    col("s.store_id") == col("st.store_id"),
    "inner"
).select(
    # Campos de Ventas
    col("s.sale_id"),
    col("s.sale_year"),
    col("s.store_id"),
    col("s.productName").alias("product_name"),
    col("s.sale_amount"),
    col("s.sale_quantity"),
    
    # Campos de Tiendas
    col("st.store_ref"),
    col("st.store_name"),
    col("st.location"),
    col("st.country"),
    col("st.store_type"),
    col("st.floor_area_sqm"),
    col("st.employee_count"),
    
    # --- Transformaciones de Negocio ---
    
    # area_category 
    when(col("st.floor_area_sqm") > 2000, "Grande")
    .when(col("st.floor_area_sqm") >= 1000, "Mediana")
    .otherwise("Pequeña").alias("area_category"),
    
    # years_diferences
    (lit(2026) - col("s.sale_year")).alias("years_diferences"),
    
    # area_per_employee 
    (col("st.floor_area_sqm") / col("st.employee_count")).alias("area_per_employee"),
    
    # sale_type 
    when(col("s.sale_quantity") > 5, "Mayorista").otherwise("Minorista").alias("sale_type"),
    
    # high_efficiency
    when((col("s.sale_amount") / col("st.floor_area_sqm")) > 1.5, "Yes").otherwise("No").alias("high_efficiency"),
    
    # Auditoría
    current_timestamp().alias("transformation_timestamp")
)

In [0]:
df_transformed.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.sales_stores_transformed")